In [1]:
# ============================================
# NOTEBOOK 1 : Traitement de Données Parquet
# Concepts : Lazy Evaluation, Transformations vs Actions
# ============================================

import sys
sys.path.insert(0, '/home/tarik/spark-workspace')

from lib import SparkJobMonitor, SparkHelper
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

# Initialiser le moniteur
monitor = SparkJobMonitor(spark)
helper = SparkHelper()

print(f"✓ Spark {spark.version}")
print(f"✓ Spark UI: http://localhost:4040")
print("="*60)

✓ Spark 3.5.0
✓ Spark UI: http://localhost:4040


In [2]:
# ============================================
# PARTIE 1 : Téléchargement d'un fichier parquet
# ============================================

!wget https://www.data.gouv.fr/api/1/datasets/r/3a80f74e-df4f-4d79-8735-4226e48526db -O adresse_france.parquet

--2026-02-17 14:22:33--  https://www.data.gouv.fr/api/1/datasets/r/3a80f74e-df4f-4d79-8735-4226e48526db
Resolving www.data.gouv.fr (www.data.gouv.fr)... 37.59.183.73, 37.59.183.91
Connecting to www.data.gouv.fr (www.data.gouv.fr)|37.59.183.73|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://static.data.gouv.fr/resources/ban-format-parquet/20250523-102223/adresses-france-10-2024.parquet [following]
--2026-02-17 14:22:33--  https://static.data.gouv.fr/resources/ban-format-parquet/20250523-102223/adresses-france-10-2024.parquet
Resolving static.data.gouv.fr (static.data.gouv.fr)... 37.59.183.91, 37.59.183.73
Connecting to static.data.gouv.fr (static.data.gouv.fr)|37.59.183.91|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 500935716 (478M) [application/octet-stream]
Saving to: ‘adresse_france.parquet’

adresse_france.parq 100%[===================>] 477,73M   782KB/s    in 15m 37s 

2026-02-17 14:38:10 (522 KB/s) - ‘adresse_fr

In [3]:
# ============================================
# PARTIE 2 : Lecture LAZY du Parquet
# ============================================

print("\n🧠 PARTIE 2 : Lecture LAZY - Aucun calcul réel")
print("="*60)

print("\n📖 CONCEPT LAZY EVALUATION:")
print("   La lecture du fichier ne charge PAS les données en mémoire")
print("   Spark crée juste un 'plan d'exécution'")

print("\n⏱️ Lecture du fichier Parquet (~500 MB)...")
start = time.time()

# TRANSFORMATION LAZY - aucune donnée n'est lue !
df_ban = spark.read.parquet("adresse_france.parquet")

elapsed = time.time() - start

print(f"✓ 'Lecture' effectuée en {elapsed:.4f}s")
print("\n💡 OBSERVATION CLÉS:")
print("   • Temps quasi-instantané (< 1 seconde)")
print("   • Aucune donnée chargée en mémoire")
print("   • Spark a juste lu les métadonnées du Parquet")
print("   • Le schéma est disponible sans lire les données")

print("\n📊 Allez dans Spark UI → Jobs")
print("   👀 Vous ne voyez AUCUN job ! C'est normal.")


🧠 PARTIE 2 : Lecture LAZY - Aucun calcul réel

📖 CONCEPT LAZY EVALUATION:
   La lecture du fichier ne charge PAS les données en mémoire
   Spark crée juste un 'plan d'exécution'

⏱️ Lecture du fichier Parquet (~500 MB)...
✓ 'Lecture' effectuée en 1.3992s

💡 OBSERVATION CLÉS:
   • Temps quasi-instantané (< 1 seconde)
   • Aucune donnée chargée en mémoire
   • Spark a juste lu les métadonnées du Parquet
   • Le schéma est disponible sans lire les données

📊 Allez dans Spark UI → Jobs
   👀 Vous ne voyez AUCUN job ! C'est normal.


In [4]:
# ============================================
# PARTIE 3 : Exploration du Schéma (LAZY)
# ============================================

print("\n📋 PARTIE 3 : Exploration du Schéma")
print("="*60)

print("\n🔍 Le schéma est disponible SANS lire les données:")
df_ban.printSchema()

print("\n📊 Colonnes disponibles:")
for i, col in enumerate(df_ban.columns, 1):
    print(f"   {i}. {col}")

print("\n💡 Ces informations viennent des métadonnées Parquet")
print("   Toujours AUCUNE donnée lue !")


📋 PARTIE 3 : Exploration du Schéma

🔍 Le schéma est disponible SANS lire les données:
root
 |-- id: string (nullable = true)
 |-- id_fantoir: string (nullable = true)
 |-- numero: long (nullable = true)
 |-- rep: string (nullable = true)
 |-- nom_voie: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- code_insee: string (nullable = true)
 |-- nom_commune: string (nullable = true)
 |-- code_insee_ancienne_commune: string (nullable = true)
 |-- nom_ancienne_commune: string (nullable = true)
 |-- type_position: string (nullable = true)
 |-- alias: string (nullable = true)
 |-- nom_ld: string (nullable = true)
 |-- libelle_acheminement: string (nullable = true)
 |-- nom_afnor: string (nullable = true)
 |-- source_position: string (nullable = true)
 |-- source_nom_voie: string (nullable = true)
 |-- certification_commune: long (nullable = true)
 |-- cad_parcelles: string (nullable = true)
 |-- geom: binary (nullable = true)
 |-- h3_7: decimal(20,0) (nullable = true)


In [5]:
# ============================================
# PARTIE 4 : Transformations LAZY (chaînage)
# ============================================

print("\n🔄 PARTIE 4 : Chaînage de TRANSFORMATIONS (lazy)")
print("="*60)

print("\n⏱️ On enchaîne plusieurs transformations...")
start = time.time()

# Toutes ces opérations sont LAZY
print("   1. Sélection de colonnes...")
df_selection = df_ban.select(
    "id",
    "numero",
    "nom_voie",
    "nom_commune",
    "code_postal"
)

print("   2. Filtrage sur Paris (75)...")
df_paris = df_selection.filter(F.col("code_postal").startswith("75"))

print("   3. Ajout d'une colonne 'adresse complète'...")
df_with_full_address = df_paris.withColumn(
    "adresse_complete",
    F.concat_ws(" ", 
                F.col("numero"), 
                F.col("nom_voie"), 
                F.col("code_postal"), 
                F.col("nom_commune"))
)

print("   4. Suppression des doublons...")
df_unique = df_with_full_address.dropDuplicates(["adresse_complete"])

print("   5. Tri par code postal...")
df_sorted = df_unique.orderBy("code_postal", "nom_voie")

elapsed = time.time() - start

print(f"\n✓ Toutes les transformations ajoutées en {elapsed:.4f}s")

print("\n💡 OBSERVATION CRITIQUE:")
print("   • 5 transformations sur un fichier de 3 GB")
print("   • Temps d'exécution : < 0.01 seconde")
print("   • RIEN n'a été calculé !")
print("   • Spark a juste construit un DAG (graphe d'exécution)")

print("\n📊 Toujours aucun job dans Spark UI !")


🔄 PARTIE 4 : Chaînage de TRANSFORMATIONS (lazy)

⏱️ On enchaîne plusieurs transformations...
   1. Sélection de colonnes...
   2. Filtrage sur Paris (75)...
   3. Ajout d'une colonne 'adresse complète'...
   4. Suppression des doublons...
   5. Tri par code postal...

✓ Toutes les transformations ajoutées en 0.1207s

💡 OBSERVATION CRITIQUE:
   • 5 transformations sur un fichier de 3 GB
   • Temps d'exécution : < 0.01 seconde
   • RIEN n'a été calculé !
   • Spark a juste construit un DAG (graphe d'exécution)

📊 Toujours aucun job dans Spark UI !


In [6]:
# ============================================
# PARTIE 5 : Première ACTION - Le réveil !
# ============================================

print("\n⚡ PARTIE 5 : Première ACTION - show()")
print("="*60)

print("\n💥 Maintenant on déclenche une ACTION : show()")
print("   TOUT le plan d'exécution va s'exécuter maintenant !")
print("\n⏳ Regardez Spark UI → Jobs (un job va ENFIN apparaître)")

result = monitor.execute_and_monitor(
    lambda: df_sorted.show(10, truncate=False),
    "JOB 1: Affichage des 10 premières adresses de Paris"
)

print("\n💡 OBSERVATIONS:")
print("   1. ✅ Un job Spark s'est exécuté")
print("   2. ✅ Les données ont été lues depuis le Parquet")
print("   3. ✅ TOUTES les transformations ont été appliquées")
print("   4. ✅ Spark a optimisé le plan (predicate pushdown)")
print("   5. ✅ Seules les colonnes nécessaires ont été lues")

print("\n🔍 Dans Spark UI → SQL/DataFrame :")
print("   • Regardez le 'Physical Plan'")
print("   • Cherchez 'PushedFilters' → filtre appliqué au niveau Parquet")
print("   • Cherchez 'ReadSchema' → colonnes réellement lues")


⚡ PARTIE 5 : Première ACTION - show()

💥 Maintenant on déclenche une ACTION : show()
   TOUT le plan d'exécution va s'exécuter maintenant !

⏳ Regardez Spark UI → Jobs (un job va ENFIN apparaître)

🔵 JOB 1: Affichage des 10 premières adresses de Paris
📌 Tracking ID: 5eab195c


+---------------------+------+-----------------------+------------------------+-----------+---------------------------------------------------------+
|id                   |numero|nom_voie               |nom_commune             |code_postal|adresse_complete                                         |
+---------------------+------+-----------------------+------------------------+-----------+---------------------------------------------------------+
|75101_ofangt_00012_s1|12    |12S1 Place du Carrousel|Paris 1er Arrondissement|75001      |12 12S1 Place du Carrousel 75001 Paris 1er Arrondissement|
|75101_adsmy0_00001_s1|1     |1S1 Place du Carrousel |Paris 1er Arrondissement|75001      |1 1S1 Place du Carrousel 75001 Paris 1er Arrondissement  |
|75101_fztbae_00002_s1|2     |2S1 Place du Carrousel |Paris 1er Arrondissement|75001      |2 2S1 Place du Carrousel 75001 Paris 1er Arrondissement  |
|75101_5paf6z_00004_w1|4     |4W1 Place du Pont Neuf |Paris 1er Arrondissement|75001      |4 4W1 Pla

In [7]:
# ============================================
# PARTIE 6 : Statistiques Globales
# ============================================

print("\n📊 PARTIE 6 : Statistiques sur la BAN Complète")
print("="*60)

print("\n1️⃣ Nombre total d'adresses en France")

result = monitor.execute_and_monitor(
    lambda: df_ban.count(),
    "JOB 2: Count sur toute la BAN"
)

print(f"\n✓ {result:,} adresses en France")

print("\n2️⃣ Nombre d'adresses par département (top 10)")

result = monitor.execute_and_monitor(
    lambda: df_ban.withColumn("departement", F.substring("code_postal", 1, 2))
        .groupBy("departement")
        .agg(F.count("*").alias("nb_adresses"))
        .orderBy(F.col("nb_adresses").desc())
        .limit(10)
        .show(truncate=False),
    "JOB 3: Top 10 départements par nombre d'adresses"
)

print("\n3️⃣ Nombre d'adresses par commune (top 20)")

result = monitor.execute_and_monitor(
    lambda: df_ban.groupBy("nom_commune", "code_postal")
        .agg(F.count("*").alias("nb_adresses"))
        .orderBy(F.col("nb_adresses").desc())
        .limit(20)
        .show(truncate=False),
    "JOB 4: Top 20 communes par nombre d'adresses"
)


📊 PARTIE 6 : Statistiques sur la BAN Complète

1️⃣ Nombre total d'adresses en France

🔵 JOB 2: Count sur toute la BAN
📌 Tracking ID: ef8828b4

✅ SUCCESS
⏱️  Durée: 0.30s
📊 Spark Job ID(s): [4, 3]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/

✓ 26,045,333 adresses en France

2️⃣ Nombre d'adresses par département (top 10)

🔵 JOB 3: Top 10 départements par nombre d'adresses
📌 Tracking ID: a4e05fe6


+-----------+-----------+
|departement|nb_adresses|
+-----------+-----------+
|59         |1014683    |
|33         |748993     |
|97         |740928     |
|62         |697765     |
|44         |572933     |
|29         |494094     |
|13         |471566     |
|76         |470813     |
|31         |460327     |
|34         |456267     |
+-----------+-----------+


✅ SUCCESS
⏱️  Durée: 1.10s
📊 Spark Job ID(s): [6, 5]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/

3️⃣ Nombre d'adresses par commune (top 20)

🔵 JOB 4: Top 20 communes par nombre d'adresses
📌 Tracking ID: 1423778e


+--------------+-----------+-----------+
|nom_commune   |code_postal|nb_adresses|
+--------------+-----------+-----------+
|Reims         |51100      |34373      |
|Brest         |29200      |32754      |
|Bordeaux      |33000      |32245      |
|Tourcoing     |59200      |30442      |
|Dijon         |21000      |29007      |
|Fort-de-France|97200      |28934      |
|Le Mans       |72000      |28574      |
|Le Tampon     |97430      |28096      |
|Niort         |79000      |27904      |
|Avignon       |84000      |26755      |
|Saint-Pierre  |97410      |26688      |
|Calais        |62100      |26263      |
|Roubaix       |59100      |26030      |
|Quimper       |29000      |25918      |
|Perpignan     |66000      |25911      |
|Montauban     |82000      |25713      |
|Narbonne      |11100      |24490      |
|Béziers       |34500      |24328      |
|Bourges       |18000      |24010      |
|Lille         |59000      |23436      |
+--------------+-----------+-----------+


✅ SUCCESS
⏱️  

In [8]:
# ============================================
# PARTIE 7 : Transformations vs Actions - Tableau
# ============================================

print("\n📚 PARTIE 7 : TRANSFORMATIONS vs ACTIONS")
print("="*60)

print("\n🔄 TRANSFORMATIONS (Lazy - ne lisent PAS les données):")
print("-" * 60)

transformations = {
    "select()": "Sélectionner des colonnes",
    "filter() / where()": "Filtrer les lignes",
    "withColumn()": "Ajouter/modifier une colonne",
    "drop()": "Supprimer une colonne",
    "groupBy()": "Grouper les données",
    "orderBy() / sort()": "Trier les données",
    "join()": "Joindre des DataFrames",
    "union()": "Concaténer des DataFrames",
    "distinct()": "Supprimer les doublons",
    "dropDuplicates()": "Supprimer les doublons sur colonnes",
    "withColumnRenamed()": "Renommer une colonne",
    "fillna()": "Remplir les valeurs nulles",
    "repartition()": "Changer le partitionnement",
}

for op, desc in transformations.items():
    print(f"   • {op:25s} → {desc}")

print("\n⚡ ACTIONS (Déclenchent la lecture et le calcul):")
print("-" * 60)

actions = {
    "show()": "Afficher les données",
    "count()": "Compter les lignes",
    "collect()": "Récupérer toutes les données",
    "take(n)": "Récupérer n lignes",
    "first()": "Récupérer la première ligne",
    "head()": "Récupérer les premières lignes",
    "describe()": "Statistiques descriptives",
    "write()": "Écrire dans un fichier",
    "foreach()": "Appliquer une fonction",
    "toPandas()": "Convertir en Pandas DataFrame",
}

for op, desc in actions.items():
    print(f"   • {op:25s} → {desc}")


📚 PARTIE 7 : TRANSFORMATIONS vs ACTIONS

🔄 TRANSFORMATIONS (Lazy - ne lisent PAS les données):
------------------------------------------------------------
   • select()                  → Sélectionner des colonnes
   • filter() / where()        → Filtrer les lignes
   • withColumn()              → Ajouter/modifier une colonne
   • drop()                    → Supprimer une colonne
   • groupBy()                 → Grouper les données
   • orderBy() / sort()        → Trier les données
   • join()                    → Joindre des DataFrames
   • union()                   → Concaténer des DataFrames
   • distinct()                → Supprimer les doublons
   • dropDuplicates()          → Supprimer les doublons sur colonnes
   • withColumnRenamed()       → Renommer une colonne
   • fillna()                  → Remplir les valeurs nulles
   • repartition()             → Changer le partitionnement

⚡ ACTIONS (Déclenchent la lecture et le calcul):
-----------------------------------------------

In [9]:
# ============================================
# PARTIE 8 : Optimisations Spark avec Parquet
# ============================================

print("\n⚙️ PARTIE 8 : Optimisations Spark sur Parquet")
print("="*60)

print("\n1️⃣ PREDICATE PUSHDOWN (filtrage au niveau stockage)")
print("-" * 60)
print("Spark applique les filtres AVANT de lire les données")

result = monitor.execute_and_monitor(
    lambda: df_ban.filter(F.col("code_postal") == "75001").count(),
    "JOB 5: Predicate pushdown - Adresses 75001"
)

print(f"\n✓ {result:,} adresses dans le 75001")
print("💡 Spark a filtré au niveau Parquet (lecture minimale)")

print("\n2️⃣ COLUMN PRUNING (lecture sélective)")
print("-" * 60)
print("Spark lit SEULEMENT les colonnes nécessaires")

result = monitor.execute_and_monitor(
    lambda: df_ban.select("nom_commune", "code_postal")
        .filter(F.col("code_postal").startswith("13"))
        .distinct()
        .count(),
    "JOB 6: Column pruning - Communes des Bouches-du-Rhône"
)

print(f"\n✓ {result} communes dans le 13")
print("💡 Spark a lu SEULEMENT 2 colonnes sur ~20")

print("\n3️⃣ ANALYSE DU PLAN D'EXÉCUTION")
print("-" * 60)

# Requête complexe
query_complexe = df_ban \
    .filter(F.col("code_postal").startswith("69")) \
    .select("nom_commune", "nom_voie") \
    .groupBy("nom_commune") \
    .agg(F.count("*").alias("nb_adresses")) \
    .orderBy(F.col("nb_adresses").desc())

print("\n📋 Plan d'exécution optimisé:")
query_complexe.explain(mode="simple")

print("\n🔍 Cherchez dans le plan:")
print("   • PushedFilters → code_postal LIKE '69%'")
print("   • ReadSchema → seulement les colonnes utilisées")


⚙️ PARTIE 8 : Optimisations Spark sur Parquet

1️⃣ PREDICATE PUSHDOWN (filtrage au niveau stockage)
------------------------------------------------------------
Spark applique les filtres AVANT de lire les données

🔵 JOB 5: Predicate pushdown - Adresses 75001
📌 Tracking ID: e8d02a9b

✅ SUCCESS
⏱️  Durée: 0.15s
📊 Spark Job ID(s): [10, 9]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/

✓ 3,309 adresses dans le 75001
💡 Spark a filtré au niveau Parquet (lecture minimale)

2️⃣ COLUMN PRUNING (lecture sélective)
------------------------------------------------------------
Spark lit SEULEMENT les colonnes nécessaires

🔵 JOB 6: Column pruning - Communes des Bouches-du-Rhône
📌 Tracking ID: 7c561b9f

✅ SUCCESS
⏱️  Durée: 0.40s
📊 Spark Job ID(s): [13, 12, 11]
📦 Stages: 6 | Tasks: 15
🌐 Spark UI: http://localhost:4040/jobs/

✓ 154 communes dans le 13
💡 Spark a lu SEULEMENT 2 colonnes sur ~20

3️⃣ ANALYSE DU PLAN D'EXÉCUTION
----------------------------------------------------------

In [10]:
# ============================================
# PARTIE 9 : Exercices Pratiques
# ============================================

print("\n🎯 PARTIE 9 : Exercices Pratiques sur la BAN")
print("="*60)

print("\nExercice 1: Les 10 noms de rues les plus fréquents en France")
print("-" * 60)

result = monitor.execute_and_monitor(
    lambda: df_ban.groupBy("nom_voie")
        .agg(F.count("*").alias("nb_occurrences"))
        .orderBy(F.col("nb_occurrences").desc())
        .limit(10)
        .show(truncate=False),
    "EXERCICE 1: Top 10 noms de rues"
)

print("\nExercice 2: Communes avec le plus de codes postaux différents")
print("-" * 60)

result = monitor.execute_and_monitor(
    lambda: df_ban.groupBy("nom_commune")
        .agg(F.countDistinct("code_postal").alias("nb_codes_postaux"))
        .filter(F.col("nb_codes_postaux") > 1)
        .orderBy(F.col("nb_codes_postaux").desc())
        .limit(10)
        .show(truncate=False),
    "EXERCICE 2: Communes multi-codes postaux"
)

print("\nExercice 3: Adresses les plus au Nord de France")
print("-" * 60)

#result = monitor.execute_and_monitor(
#    lambda: df_ban.select("nom_commune", "nom_voie", "numero", "position_latitude")
#        .filter(F.col("position_latitude").isNotNull())
#        .orderBy(F.col("position_latitude").desc())
#        .limit(10)
#        .show(truncate=False),
#    "EXERCICE 3: Adresses au Nord"
#)

print("\nExercice 4: Statistiques par région (via code postal)")
print("-" * 60)

result = monitor.execute_and_monitor(
    lambda: df_ban.withColumn("dept", F.substring("code_postal", 1, 2))
        .groupBy("dept")
        .agg(
            F.count("*").alias("nb_adresses"),
            F.countDistinct("nom_commune").alias("nb_communes"),
            F.countDistinct("nom_voie").alias("nb_voies_uniques")
        )
        .orderBy(F.col("nb_adresses").desc())
        .limit(15)
        .show(truncate=False),
    "EXERCICE 4: Stats par département"
)


🎯 PARTIE 9 : Exercices Pratiques sur la BAN

Exercice 1: Les 10 noms de rues les plus fréquents en France
------------------------------------------------------------

🔵 EXERCICE 1: Top 10 noms de rues
📌 Tracking ID: 773bafc5


+--------------------+--------------+
|nom_voie            |nb_occurrences|
+--------------------+--------------+
|Grande Rue          |154834        |
|Rue Principale      |81117         |
|Rue de l'Eglise     |60429         |
|Rue de la Gare      |54014         |
|Rue Pasteur         |53482         |
|Rue du Moulin       |53281         |
|Rue de la République|48979         |
|Rue Victor Hugo     |48773         |
|Rue de la Mairie    |45573         |
|Rue Jean Jaurès     |39473         |
+--------------------+--------------+


✅ SUCCESS
⏱️  Durée: 1.86s
📊 Spark Job ID(s): [15, 14]
📦 Stages: 3 | Tasks: 13
🌐 Spark UI: http://localhost:4040/jobs/

Exercice 2: Communes avec le plus de codes postaux différents
------------------------------------------------------------

🔵 EXERCICE 2: Communes multi-codes postaux
📌 Tracking ID: 68b22dda


+--------------+----------------+
|nom_commune   |nb_codes_postaux|
+--------------+----------------+
|Saint-Paul    |13              |
|Sainte-Colombe|12              |
|Saint-Sauveur |11              |
|Beaulieu      |10              |
|Saint-Pierre  |10              |
|Saint-Aubin   |10              |
|Le Pin        |9               |
|Saint-Marcel  |9               |
|Haguenau      |9               |
|Saint-Michel  |9               |
+--------------+----------------+


✅ SUCCESS
⏱️  Durée: 1.07s
📊 Spark Job ID(s): [18, 17, 16]
📦 Stages: 6 | Tasks: 15
🌐 Spark UI: http://localhost:4040/jobs/

Exercice 3: Adresses les plus au Nord de France
------------------------------------------------------------

Exercice 4: Statistiques par région (via code postal)
------------------------------------------------------------

🔵 EXERCICE 4: Stats par département
📌 Tracking ID: a22b7d18


[Stage 31:===========================================>              (3 + 1) / 4]

+----+-----------+-----------+----------------+
|dept|nb_adresses|nb_communes|nb_voies_uniques|
+----+-----------+-----------+----------------+
|59  |1014683    |648        |25865           |
|33  |748993     |537        |40245           |
|97  |740928     |123        |30985           |
|62  |697765     |890        |20430           |
|44  |572933     |207        |32953           |
|29  |494094     |277        |37397           |
|13  |471566     |135        |26593           |
|76  |470813     |708        |20849           |
|31  |460327     |586        |28335           |
|34  |456267     |342        |25052           |
|77  |454458     |507        |19349           |
|17  |432358     |463        |25990           |
|35  |429328     |332        |34849           |
|38  |426109     |512        |27219           |
|85  |423393     |252        |25365           |
+----+-----------+-----------+----------------+


✅ SUCCESS
⏱️  Durée: 4.45s
📊 Spark Job ID(s): [21, 20, 19]
📦 Stages: 6 | Tasks: 21
🌐 S

In [11]:
# ============================================
# PARTIE 10 : Cache et Performance
# ============================================

print("\n🚀 PARTIE 10 : Impact du Cache sur la Performance")
print("="*60)

# Créer un DataFrame filtré
df_idf = df_ban.filter(
    F.col("code_postal").startswith("75") |
    F.col("code_postal").startswith("77") |
    F.col("code_postal").startswith("78") |
    F.col("code_postal").startswith("91") |
    F.col("code_postal").startswith("92") |
    F.col("code_postal").startswith("93") |
    F.col("code_postal").startswith("94") |
    F.col("code_postal").startswith("95")
)

print("\n1️⃣ Première exécution SANS cache")
result1 = monitor.execute_and_monitor(
    lambda: df_idf.count(),
    "JOB 7: Count Île-de-France (sans cache)"
)

print(f"\n✓ {result1:,} adresses en Île-de-France")

print("\n2️⃣ Mise en cache")
df_idf.cache()

print("\n3️⃣ Deuxième exécution AVEC cache (premier passage)")
result2 = monitor.execute_and_monitor(
    lambda: df_idf.count(),
    "JOB 8: Count Île-de-France (mise en cache)"
)

print("\n4️⃣ Troisième exécution AVEC cache (depuis le cache)")
result3 = monitor.execute_and_monitor(
    lambda: df_idf.count(),
    "JOB 9: Count Île-de-France (depuis cache)"
)

print("\n💡 OBSERVATIONS:")
print("   • JOB 7 : Lecture depuis Parquet")
print("   • JOB 8 : Lecture + mise en cache")
print("   • JOB 9 : Lecture DEPUIS le cache (beaucoup plus rapide !)")

print("\n🌐 Dans Spark UI → Storage:")
print("   • Vous voyez les données en cache")
print("   • Taille mémoire utilisée")
print("   • Nombre de partitions")

# Libérer le cache
df_idf.unpersist()
print("\n✓ Cache libéré")


🚀 PARTIE 10 : Impact du Cache sur la Performance

1️⃣ Première exécution SANS cache

🔵 JOB 7: Count Île-de-France (sans cache)
📌 Tracking ID: b6dd10ca

✅ SUCCESS
⏱️  Durée: 0.28s
📊 Spark Job ID(s): [23, 22]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/

✓ 2,186,664 adresses en Île-de-France

2️⃣ Mise en cache

3️⃣ Deuxième exécution AVEC cache (premier passage)

🔵 JOB 8: Count Île-de-France (mise en cache)
📌 Tracking ID: 4faf0609


[Stage 40:===========================================>              (3 + 1) / 4]


✅ SUCCESS
⏱️  Durée: 7.24s
📊 Spark Job ID(s): [26, 25, 24]
📦 Stages: 4 | Tasks: 13
🌐 Spark UI: http://localhost:4040/jobs/

4️⃣ Troisième exécution AVEC cache (depuis le cache)

🔵 JOB 9: Count Île-de-France (depuis cache)
📌 Tracking ID: 57457d5a

✅ SUCCESS
⏱️  Durée: 0.08s
📊 Spark Job ID(s): [28, 27]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/

💡 OBSERVATIONS:
   • JOB 7 : Lecture depuis Parquet
   • JOB 8 : Lecture + mise en cache
   • JOB 9 : Lecture DEPUIS le cache (beaucoup plus rapide !)

🌐 Dans Spark UI → Storage:
   • Vous voyez les données en cache
   • Taille mémoire utilisée
   • Nombre de partitions

✓ Cache libéré


In [12]:
# ============================================
# PARTIE 11 : Repartitionnement et Écriture
# ============================================

print("\n💾 PARTIE 11 : Repartitionnement et Écriture Parquet")
print("="*60)

# Filtrer sur une région
df_bretagne = df_ban.filter(
    F.col("code_postal").startswith("22") |
    F.col("code_postal").startswith("29") |
    F.col("code_postal").startswith("35") |
    F.col("code_postal").startswith("56")
).withColumn("departement", F.substring("code_postal", 1, 2))

output_path = "/tmp/spark_demo/ban_bretagne_parquet"

print("\n1️⃣ Écriture partitionnée par département")

result = monitor.execute_and_monitor(
    lambda: df_bretagne.write
        .mode("overwrite")
        .partitionBy("departement")
        .parquet(output_path),
    "JOB 10: Écriture Parquet partitionnée (Bretagne)"
)

print(f"\n✓ Données écrites dans: {output_path}")

print("\n📁 Structure des partitions:")
for root, dirs, files in os.walk(output_path):
    level = root.replace(output_path, '').count(os.sep)
    indent = ' ' * 2 * level
    folder = os.path.basename(root)
    if folder:
        print(f'{indent}{folder}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for file in files[:3]:
            size = os.path.getsize(os.path.join(root, file)) / (1024**2)
            print(f'{subindent}{file} ({size:.2f} MB)')
        if len(files) > 3:
            print(f'{subindent}... et {len(files) - 3} autres fichiers')

print("\n2️⃣ Lecture avec partition pruning")

result = monitor.execute_and_monitor(
    lambda: spark.read.parquet(output_path)
        .filter(F.col("departement") == "29")
        .count(),
    "JOB 11: Lecture avec partition pruning (Finistère)"
)

print(f"\n✓ {result:,} adresses dans le Finistère (29)")
print("💡 Spark a lu SEULEMENT le dossier departement=29/")


💾 PARTIE 11 : Repartitionnement et Écriture Parquet

1️⃣ Écriture partitionnée par département

🔵 JOB 10: Écriture Parquet partitionnée (Bretagne)
📌 Tracking ID: e84929b4



✅ SUCCESS
⏱️  Durée: 3.52s
📊 Spark Job ID(s): [29]
📦 Stages: 1 | Tasks: 4
🌐 Spark UI: http://localhost:4040/jobs/

✓ Données écrites dans: /tmp/spark_demo/ban_bretagne_parquet

📁 Structure des partitions:
ban_bretagne_parquet/
  ._SUCCESS.crc (0.00 MB)
  _SUCCESS (0.00 MB)
  departement=56/
    .part-00002-7109cbe2-ff6a-4c02-b0c3-3d9422bc6e9d.c000.snappy.parquet.crc (0.09 MB)
    part-00002-7109cbe2-ff6a-4c02-b0c3-3d9422bc6e9d.c000.snappy.parquet (11.84 MB)
  departement=22/
    .part-00000-7109cbe2-ff6a-4c02-b0c3-3d9422bc6e9d.c000.snappy.parquet.crc (0.08 MB)
    part-00000-7109cbe2-ff6a-4c02-b0c3-3d9422bc6e9d.c000.snappy.parquet (10.52 MB)
  departement=29/
    .part-00000-7109cbe2-ff6a-4c02-b0c3-3d9422bc6e9d.c000.snappy.parquet.crc (0.11 MB)
    part-00000-7109cbe2-ff6a-4c02-b0c3-3d9422bc6e9d.c000.snappy.parquet (14.06 MB)
  departement=35/
    .part-00001-7109cbe2-ff6a-4c02-b0c3-3d9422bc6e9d.c000.snappy.parquet.crc (0.10 MB)
    part-00001-7109cbe2-ff6a-4c02-b0c3-3d9422bc6e9d.c000

In [13]:
# ============================================
# PARTIE 11 BIS : Comprendre le Partitionnement en Profondeur
# ============================================

print("\n🔍 PARTIE 11 BIS : Le Partitionnement Parquet Expliqué en Détail")
print("="*70)

print("""
📚 QU'EST-CE QUE LE PARTITIONNEMENT ?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Le partitionnement divise les données en sous-dossiers selon les valeurs
d'une ou plusieurs colonnes.

Structure classique:
    data/
    ├── departement=75/
    │   ├── part-00000-xxx.snappy.parquet
    │   └── part-00001-xxx.snappy.parquet
    ├── departement=92/
    │   └── part-00000-xxx.snappy.parquet
    └── _SUCCESS

Avantages:
  ✓ PARTITION PRUNING : Spark lit seulement les dossiers nécessaires
  ✓ PERFORMANCE : Évite de scanner toutes les données
  ✓ ORGANISATION : Structure logique et claire
  ✓ COMPATIBILITÉ : Standard Hive/Hadoop
""")

# ============================================
# 1. EXPÉRIMENTATION : Impact du Partitionnement
# ============================================

print("\n" + "="*70)
print("1️⃣ EXPÉRIMENTATION : Partitionné vs Non-Partitionné")
print("="*70)

# Préparer les données
df_sample = df_ban.filter(
    F.col("code_postal").startswith("75") |
    F.col("code_postal").startswith("92") |
    F.col("code_postal").startswith("93")
).withColumn("departement", F.substring("code_postal", 1, 2))

# Chemin 1 : NON partitionné
path_non_partitioned = "/tmp/spark_demo/ban_non_partitioned"

print("\n📝 Écriture NON partitionnée...")
result = monitor.execute_and_monitor(
    lambda: df_sample.write.mode("overwrite").parquet(path_non_partitioned),
    "ÉCRITURE : Sans partitionnement"
)

# Chemin 2 : Partitionné
path_partitioned = "/tmp/spark_demo/ban_partitioned"

print("\n📝 Écriture PARTITIONNÉE par département...")
result = monitor.execute_and_monitor(
    lambda: df_sample.write.mode("overwrite")
        .partitionBy("departement")
        .parquet(path_partitioned),
    "ÉCRITURE : Avec partitionnement"
)

# ============================================
# 2. ANALYSE DES FICHIERS GÉNÉRÉS
# ============================================

print("\n" + "="*70)
print("2️⃣ ANALYSE DES FICHIERS GÉNÉRÉS")
print("="*70)

import subprocess
import glob

print("\n📁 Structure NON partitionnée:")
print("-" * 70)
for item in os.listdir(path_non_partitioned)[:10]:
    item_path = os.path.join(path_non_partitioned, item)
    if os.path.isfile(item_path):
        size = os.path.getsize(item_path)
        print(f"   {item:60s} {size:>12,} bytes")

print("\n📁 Structure PARTITIONNÉE:")
print("-" * 70)
for root, dirs, files in os.walk(path_partitioned):
    level = root.replace(path_partitioned, '').count(os.sep)
    indent = '   ' * level
    folder = os.path.basename(root)
    if folder:
        print(f"{indent}📂 {folder}/")
    
    if files and level < 3:
        for file in files:
            file_path = os.path.join(root, file)
            size = os.path.getsize(file_path)
            print(f"{indent}   📄 {file:50s} {size:>12,} bytes")

# ============================================
# 3. LES FICHIERS .snappy.parquet et .crc
# ============================================

print("\n" + "="*70)
print("3️⃣ COMPRENDRE LES FICHIERS .snappy.parquet et .crc")
print("="*70)

print("""
📄 TYPES DE FICHIERS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. part-00000-xxx.snappy.parquet
   ├─ FORMAT : Parquet (format colonnaire binaire)
   ├─ COMPRESSION : Snappy (rapide, ratio moyen)
   ├─ CONTENU : VOS DONNÉES
   └─ LECTURE : Lisible par Spark, Pandas, DuckDB, etc.

2. .part-00000-xxx.snappy.parquet.crc
   ├─ TYPE : Fichier de checksum (CRC32)
   ├─ TAILLE : Très petit (~12 bytes)
   ├─ RÔLE : Vérification d'intégrité HDFS
   ├─ LECTURE : Utilisé automatiquement par Hadoop/Spark
   └─ SUPPRESSION : Peut être supprimé hors environnement Hadoop

3. _SUCCESS
   ├─ TYPE : Marqueur de succès (fichier vide)
   ├─ TAILLE : 0 bytes
   ├─ RÔLE : Indique que l'écriture est terminée avec succès
   └─ UTILITÉ : Évite de lire des données partielles

4. _committed_xxx / _started_xxx
   ├─ TYPE : Fichiers temporaires
   ├─ RÔLE : Coordination des tasks Spark
   └─ NETTOYAGE : Supprimés automatiquement en cas de succès
""")

# Exemple concret
print("\n🔍 EXEMPLE CONCRET :")
print("-" * 70)

parquet_files = glob.glob(f"{path_partitioned}/**/*.parquet", recursive=True)
if parquet_files:
    example_file = parquet_files[0]
    example_base = os.path.basename(example_file)
    example_dir = os.path.dirname(example_file)
    
    print(f"\nFichier Parquet:")
    print(f"   📄 {example_base}")
    print(f"   📊 Taille : {os.path.getsize(example_file):,} bytes")
    
    # Chercher le .crc correspondant
    crc_file = os.path.join(example_dir, f".{example_base}.crc")
    if os.path.exists(crc_file):
        print(f"\nFichier CRC associé:")
        print(f"   🔐 .{example_base}.crc")
        print(f"   📊 Taille : {os.path.getsize(crc_file):,} bytes")
        print(f"   💡 Rôle : Vérifier l'intégrité du fichier Parquet")

# ============================================
# 4. COMPRESSIONS DISPONIBLES
# ============================================

print("\n" + "="*70)
print("4️⃣ COMPRESSIONS PARQUET DISPONIBLES")
print("="*70)

print("""
🗜️ CODECS DE COMPRESSION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Codec      | Vitesse  | Ratio | CPU    | Cas d'usage
-----------|----------|-------|--------|--------------------------------
SNAPPY     | ⚡⚡⚡    | ⭐⭐   | Faible | ✅ PAR DÉFAUT - Équilibré
GZIP       | ⚡       | ⭐⭐⭐  | Élevé  | Stockage long terme, bande passante limitée
LZ4        | ⚡⚡⚡⚡  | ⭐     | Faible | Vitesse maximale
ZSTD       | ⚡⚡     | ⭐⭐⭐⭐| Moyen  | Meilleur compromis moderne (Spark 3.2+)
BROTLI     | ⚡       | ⭐⭐⭐⭐| Élevé  | Meilleure compression
UNCOMPRESSED| ⚡⚡⚡⚡ | -     | Aucun  | Debug, pas recommandé

RECOMMANDATIONS :
  • Développement/Tests : SNAPPY (défaut)
  • Production (lecture intensive) : LZ4 ou SNAPPY
  • Stockage long terme : ZSTD ou GZIP
  • Bande passante limitée : GZIP ou ZSTD
""")

# Démonstration avec différentes compressions
compressions = ["snappy", "gzip", "uncompressed"]
compression_stats = []

for codec in compressions:
    path = f"/tmp/spark_demo/ban_compression_{codec}"
    
    print(f"\n📝 Test compression : {codec.upper()}")
    
    result = monitor.execute_and_monitor(
        lambda c=codec, p=path: df_sample.limit(10000).write
            .mode("overwrite")
            .option("compression", c)
            .parquet(p),
        f"ÉCRITURE : Compression {codec.upper()}"
    )
    
    # Calculer la taille totale
    total_size = sum(
        os.path.getsize(os.path.join(root, file))
        for root, _, files in os.walk(path)
        for file in files
        if file.endswith('.parquet')
    )
    
    compression_stats.append({
        'codec': codec,
        'size_mb': total_size / (1024**2)
    })
    
    print(f"   💾 Taille : {total_size / (1024**2):.2f} MB")

print("\n📊 COMPARAISON DES COMPRESSIONS (10k lignes):")
print("-" * 70)
for stat in sorted(compression_stats, key=lambda x: x['size_mb']):
    print(f"   {stat['codec']:15s} : {stat['size_mb']:8.2f} MB")

# ============================================
# 5. VERSIONS SPARK ET COMPATIBILITÉ
# ============================================

print("\n" + "="*70)
print("5️⃣ VERSIONS SPARK ET COMPATIBILITÉ")
print("="*70)

print(f"""
⚠️ ATTENTION AUX VERSIONS !
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Version Spark actuelle : {spark.version}

PROBLÈMES DE COMPATIBILITÉ :

1. FORMAT PARQUET
   ├─ Spark 2.x : Parquet 1.10.x
   ├─ Spark 3.0-3.1 : Parquet 1.10.x - 1.11.x
   ├─ Spark 3.2+ : Parquet 1.12.x+
   └─ ⚠️ Les anciens Spark peuvent ne pas lire les nouveaux formats

2. COMPRESSION ZSTD
   ├─ Disponible : Spark 3.2+
   └─ ❌ Spark 2.x et 3.0/3.1 : erreur si données en ZSTD

3. TIMESTAMP
   ├─ Spark 2.x : INT96 (ancien format)
   ├─ Spark 3.0+ : INT64 (nouveau format, par défaut)
   └─ ⚠️ Problèmes de lecture croisée possible

4. COLONNES NESTED
   ├─ Amélioration constante du support
   └─ Vérifier la compatibilité pour structures complexes

BONNES PRATIQUES POUR LE PARTAGE :

✅ 1. Documenter la version Spark utilisée
   README.md : "Généré avec Spark 3.5.0"

✅ 2. Utiliser des compressions compatibles
   SNAPPY ou GZIP = compatibilité maximale
   ZSTD = Spark 3.2+ uniquement

✅ 3. Tester avec la version cible
   Si destinataire utilise Spark 2.x, tester !

✅ 4. Configurer le format timestamp (si nécessaire)
   .option("parquet.writer.version", "v1")  # Ancien format
   .option("parquet.writer.version", "v2")  # Nouveau format

✅ 5. Éviter les features récentes si partage
   Rester sur des formats standards
""")

# Démonstration : écrire avec version compatible
print("\n🔧 EXEMPLE : Écriture compatible Spark 2.x/3.x")
print("-" * 70)

path_compatible = "/tmp/spark_demo/ban_compatible"

result = monitor.execute_and_monitor(
    lambda: df_sample.limit(1000).write
        .mode("overwrite")
        .option("compression", "snappy")  # Compatible partout
        .option("parquet.writer.version", "v1")  # Format ancien
        .partitionBy("departement")
        .parquet(path_compatible),
    "ÉCRITURE : Format compatible"
)

print("✅ Fichiers lisibles par Spark 2.x, 3.x, Pandas, DuckDB, etc.")

# ============================================
# 6. PERFORMANCE : Lecture Partitionnée vs Non-Partitionnée
# ============================================

print("\n" + "="*70)
print("6️⃣ PERFORMANCE : Partition Pruning en Action")
print("="*70)

print("\n🔍 Test 1 : Lecture d'UN département depuis fichier NON partitionné")
print("-" * 70)

result1 = monitor.execute_and_monitor(
    lambda: spark.read.parquet(path_non_partitioned)
        .filter(F.col("departement") == "75")
        .count(),
    "LECTURE : Dept 75 (non partitionné)"
)

print(f"✓ {result1:,} adresses trouvées")
print("💡 Spark a dû SCANNER TOUT LE FICHIER")

print("\n🔍 Test 2 : Lecture d'UN département depuis fichier PARTITIONNÉ")
print("-" * 70)

result2 = monitor.execute_and_monitor(
    lambda: spark.read.parquet(path_partitioned)
        .filter(F.col("departement") == "75")
        .count(),
    "LECTURE : Dept 75 (partitionné)"
)

print(f"✓ {result2:,} adresses trouvées")
print("💡 Spark a lu SEULEMENT le dossier departement=75/")
print("⚡ BEAUCOUP PLUS RAPIDE !")

# Vérifier avec explain
print("\n📋 Plan d'exécution (partitionné):")
print("-" * 70)
spark.read.parquet(path_partitioned) \
    .filter(F.col("departement") == "75") \
    .explain()

print("\n🔍 Cherchez 'PartitionFilters' dans le plan ci-dessus")
print("   C'est la preuve du partition pruning !")

# ============================================
# 7. BONNES PRATIQUES DE PARTITIONNEMENT
# ============================================

print("\n" + "="*70)
print("7️⃣ BONNES PRATIQUES DE PARTITIONNEMENT")
print("="*70)

print("""
✅ QUAND PARTITIONNER ?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

OUI, partitionner si :
  ✓ Filtrage fréquent sur une colonne (ex: date, région, catégorie)
  ✓ Cardinalité moyenne (10 à 10,000 partitions idéal)
  ✓ Données volumineuses (> 1 GB)
  ✓ Lecture sélective courante

NON, ne pas partitionner si :
  ✗ Cardinalité trop haute (> 100,000 partitions)
  ✗ Cardinalité trop basse (< 5 partitions)
  ✗ Scan complet systématique
  ✗ Données petites (< 100 MB)

⚠️ ANTI-PATTERNS À ÉVITER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❌ Partitionner par ID utilisateur (millions de partitions)
   → Too many small files problem
   → Metadata overhead
   → Performance dégradée

❌ Partitionner par timestamp exact
   → Une partition par seconde = explosion
   → Utiliser date ou heure plutôt

❌ Créer des partitions < 100 MB
   → Overhead > bénéfice
   → Privilégier des partitions 128 MB - 1 GB

❌ Trop de niveaux de partitionnement
   partitionBy("year", "month", "day", "hour", "country", "category")
   → Complexité excessive
   → Max 2-3 niveaux recommandés

✅ EXEMPLES VALIDES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Logs applicatifs :
  .partitionBy("date", "level")  # 365 dates × 5 levels = ~1,800 partitions

E-commerce :
  .partitionBy("year_month", "category")  # 12 mois × 10 catégories = 120

Données géographiques :
  .partitionBy("country", "region")  # 200 pays × 5 régions = ~1,000

Événements :
  .partitionBy("event_date")  # 1 partition par jour

⚙️ CONFIGURATION OPTIMALE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Taille cible des fichiers Parquet
spark.conf.set("spark.sql.files.maxRecordsPerFile", 1000000)
spark.conf.set("spark.sql.files.maxPartitionBytes", 134217728)  # 128 MB

# Nombre de fichiers par partition
df.repartition(4, "departement").write.partitionBy("departement").parquet(...)
   → 4 fichiers par partition (si assez de données)

# Coalescer pour réduire le nombre de fichiers
df.coalesce(1).write.partitionBy("date").parquet(...)
   → 1 seul fichier par partition
""")

# ============================================
# 8. DEBUGGING : Nombre de Fichiers
# ============================================

print("\n" + "="*70)
print("8️⃣ CONTRÔLER LE NOMBRE DE FICHIERS PAR PARTITION")
print("="*70)

# Créer des exemples
path_many_files = "/tmp/spark_demo/ban_many_files"
path_few_files = "/tmp/spark_demo/ban_few_files"

print("\n📝 Test 1 : Beaucoup de petits fichiers (par défaut)")

result = monitor.execute_and_monitor(
    lambda: df_sample.write.mode("overwrite")
        .partitionBy("departement")
        .parquet(path_many_files),
    "ÉCRITURE : Partitions par défaut"
)

# Compter les fichiers
for dept in ["75", "92", "93"]:
    dept_path = os.path.join(path_many_files, f"departement={dept}")
    if os.path.exists(dept_path):
        parquet_count = len([f for f in os.listdir(dept_path) if f.endswith('.parquet')])
        print(f"   Département {dept} : {parquet_count} fichiers .parquet")

print("\n📝 Test 2 : Un fichier par partition (coalesce)")

result = monitor.execute_and_monitor(
    lambda: df_sample
        .repartition("departement")  # Regrouper par département
        .coalesce(3)  # 1 fichier par partition
        .write.mode("overwrite")
        .partitionBy("departement")
        .parquet(path_few_files),
    "ÉCRITURE : 1 fichier par partition"
)

# Compter les fichiers
for dept in ["75", "92", "93"]:
    dept_path = os.path.join(path_few_files, f"departement={dept}")
    if os.path.exists(dept_path):
        parquet_count = len([f for f in os.listdir(dept_path) if f.endswith('.parquet')])
        print(f"   Département {dept} : {parquet_count} fichiers .parquet")

print("""
💡 RÈGLE D'OR :
   1 fichier par partition = Plus simple à gérer
   Mais : écriture moins parallélisée
   
   Plusieurs fichiers = Meilleure parallélisation
   Mais : plus de fichiers à gérer
   
   Compromis recommandé : 2-8 fichiers par partition
""")

# ============================================
# RÉSUMÉ FINAL
# ============================================

print("\n" + "="*70)
print("📚 RÉSUMÉ : PARTITIONNEMENT PARQUET")
print("="*70)

print("""
🎯 POINTS CLÉS À RETENIR

1️⃣ FICHIERS
   • .parquet : VOS DONNÉES (format colonnaire compressé)
   • .crc : Checksum HDFS (intégrité, peut être supprimé hors Hadoop)
   • _SUCCESS : Marqueur de succès (fichier vide)

2️⃣ COMPRESSION
   • SNAPPY : Défaut, équilibré, compatible partout ✅
   • GZIP : Meilleure compression, plus lent
   • ZSTD : Meilleur compromis moderne (Spark 3.2+)
   • LZ4 : Plus rapide, moins de compression

3️⃣ PARTITIONNEMENT
   • Organise les données en sous-dossiers
   • Permet le PARTITION PRUNING (lecture sélective)
   • Idéal : 10 à 10,000 partitions
   • Éviter : cardinalité trop haute ou trop basse

4️⃣ COMPATIBILITÉ
   • Documenter la version Spark utilisée
   • SNAPPY + Parquet v1 = compatibilité maximale
   • ZSTD = Spark 3.2+ uniquement
   • Tester avec la version cible !

5️⃣ PERFORMANCE
   • Partitionné >> Non-partitionné pour lectures sélectives
   • Taille cible : 128 MB - 1 GB par fichier
   • 2-8 fichiers par partition = bon compromis

6️⃣ PARTAGE DE DONNÉES
   ✅ Utiliser compression standard (snappy/gzip)
   ✅ Documenter : version Spark, schéma, partitions
   ✅ Inclure _SUCCESS pour indiquer complétude
   ✅ Tester la lecture avec version destinataire
""")

print("\n✅ Section terminée ! Vous maîtrisez maintenant le partitionnement Parquet.")


🔍 PARTIE 11 BIS : Le Partitionnement Parquet Expliqué en Détail

📚 QU'EST-CE QUE LE PARTITIONNEMENT ?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Le partitionnement divise les données en sous-dossiers selon les valeurs
d'une ou plusieurs colonnes.

Structure classique:
    data/
    ├── departement=75/
    │   ├── part-00000-xxx.snappy.parquet
    │   └── part-00001-xxx.snappy.parquet
    ├── departement=92/
    │   └── part-00000-xxx.snappy.parquet
    └── _SUCCESS

Avantages:
  ✓ PARTITION PRUNING : Spark lit seulement les dossiers nécessaires
  ✓ PERFORMANCE : Évite de scanner toutes les données
  ✓ ORGANISATION : Structure logique et claire
  ✓ COMPATIBILITÉ : Standard Hive/Hadoop


1️⃣ EXPÉRIMENTATION : Partitionné vs Non-Partitionné

📝 Écriture NON partitionnée...

🔵 ÉCRITURE : Sans partitionnement
📌 Tracking ID: 6db678ca



✅ SUCCESS
⏱️  Durée: 1.60s
📊 Spark Job ID(s): [33]
📦 Stages: 1 | Tasks: 4
🌐 Spark UI: http://localhost:4040/jobs/

📝 Écriture PARTITIONNÉE par département...

🔵 ÉCRITURE : Avec partitionnement
📌 Tracking ID: 922dfd84



✅ SUCCESS
⏱️  Durée: 1.76s
📊 Spark Job ID(s): [34]
📦 Stages: 1 | Tasks: 4
🌐 Spark UI: http://localhost:4040/jobs/

2️⃣ ANALYSE DES FICHIERS GÉNÉRÉS

📁 Structure NON partitionnée:
----------------------------------------------------------------------
   ._SUCCESS.crc                                                           8 bytes
   part-00003-bc6b51f8-9c54-40ff-8121-b73b5fe0db97-c000.snappy.parquet   13,169,328 bytes
   .part-00000-bc6b51f8-9c54-40ff-8121-b73b5fe0db97-c000.snappy.parquet.crc           28 bytes
   .part-00003-bc6b51f8-9c54-40ff-8121-b73b5fe0db97-c000.snappy.parquet.crc      102,896 bytes
   part-00000-bc6b51f8-9c54-40ff-8121-b73b5fe0db97-c000.snappy.parquet        2,428 bytes
   _SUCCESS                                                                0 bytes

📁 Structure PARTITIONNÉE:
----------------------------------------------------------------------
📂 ban_partitioned/
   📄 ._SUCCESS.crc                                                 8 bytes
   📄 _SUCCESS        


✅ SUCCESS
⏱️  Durée: 1.59s
📊 Spark Job ID(s): [50]
📦 Stages: 1 | Tasks: 4
🌐 Spark UI: http://localhost:4040/jobs/
   Département 75 : 1 fichiers .parquet
   Département 92 : 1 fichiers .parquet
   Département 93 : 1 fichiers .parquet

📝 Test 2 : Un fichier par partition (coalesce)

🔵 ÉCRITURE : 1 fichier par partition
📌 Tracking ID: 59cab454


[Stage 78:======================================>                   (2 + 1) / 3]


✅ SUCCESS
⏱️  Durée: 1.72s
📊 Spark Job ID(s): [52, 51]
📦 Stages: 3 | Tasks: 11
🌐 Spark UI: http://localhost:4040/jobs/
   Département 75 : 1 fichiers .parquet
   Département 92 : 1 fichiers .parquet
   Département 93 : 1 fichiers .parquet

💡 RÈGLE D'OR :
   1 fichier par partition = Plus simple à gérer
   Mais : écriture moins parallélisée

   Plusieurs fichiers = Meilleure parallélisation
   Mais : plus de fichiers à gérer

   Compromis recommandé : 2-8 fichiers par partition


📚 RÉSUMÉ : PARTITIONNEMENT PARQUET

🎯 POINTS CLÉS À RETENIR

1️⃣ FICHIERS
   • .parquet : VOS DONNÉES (format colonnaire compressé)
   • .crc : Checksum HDFS (intégrité, peut être supprimé hors Hadoop)
   • _SUCCESS : Marqueur de succès (fichier vide)

2️⃣ COMPRESSION
   • SNAPPY : Défaut, équilibré, compatible partout ✅
   • GZIP : Meilleure compression, plus lent
   • ZSTD : Meilleur compromis moderne (Spark 3.2+)
   • LZ4 : Plus rapide, moins de compression

3️⃣ PARTITIONNEMENT
   • Organise les données en 

In [15]:
# ============================================
# RÉSUMÉ ET HISTORIQUE
# ============================================

print("\n" + "="*60)
print("📚 RÉSUMÉ DES CONCEPTS CLÉS")
print("="*60)

print("""
🎯 LAZY EVALUATION
   ✓ Les transformations ne s'exécutent PAS immédiatement
   ✓ Spark construit un DAG (graphe d'exécution)
   ✓ Les actions déclenchent l'exécution complète
   ✓ Permet d'optimiser avant de calculer

🔄 TRANSFORMATIONS (lazy)
   • select, filter, withColumn, groupBy, join
   • Retournent un nouveau DataFrame
   • AUCUNE donnée lue ou calculée
   • S'enchaînent instantanément

⚡ ACTIONS (eager - déclenchent l'exécution)
   • show, count, collect, write
   • Exécutent TOUT le plan d'exécution
   • Lisent et calculent les données
   • Retournent un résultat

⚙️ OPTIMISATIONS SPARK
   ✓ Predicate Pushdown : filtres au niveau Parquet
   ✓ Column Pruning : lecture sélective de colonnes
   ✓ Partition Pruning : lecture sélective de dossiers
   ✓ Catalyst Optimizer : réorganise le plan

📦 PARQUET
   ✓ Format colonnaire (parfait pour l'analytique)
   ✓ Compression efficace (gain 5-10x vs CSV)
   ✓ Métadonnées riches (schéma, stats)
   ✓ Partitionnement natif
   ✓ Support predicate pushdown
""")

# Afficher l'historique complet
monitor.show_history()

# Statistiques
stats = monitor.get_job_stats()
print(f"\n📊 STATISTIQUES GLOBALES:")
print(f"   Total jobs exécutés : {stats['total_jobs']}")
print(f"   Jobs réussis : {stats['success']}")
print(f"   Jobs échoués : {stats['failed']}")
print(f"   Durée totale : {stats['total_duration']:.2f}s")
print(f"   Durée moyenne : {stats['avg_duration']:.2f}s")

print("\n✅ Notebook terminé !")
print("🌐 Explorez Spark UI pour approfondir votre compréhension")


📚 RÉSUMÉ DES CONCEPTS CLÉS

🎯 LAZY EVALUATION
   ✓ Les transformations ne s'exécutent PAS immédiatement
   ✓ Spark construit un DAG (graphe d'exécution)
   ✓ Les actions déclenchent l'exécution complète
   ✓ Permet d'optimiser avant de calculer

🔄 TRANSFORMATIONS (lazy)
   • select, filter, withColumn, groupBy, join
   • Retournent un nouveau DataFrame
   • AUCUNE donnée lue ou calculée
   • S'enchaînent instantanément

⚡ ACTIONS (eager - déclenchent l'exécution)
   • show, count, collect, write
   • Exécutent TOUT le plan d'exécution
   • Lisent et calculent les données
   • Retournent un résultat

⚙️ OPTIMISATIONS SPARK
   ✓ Predicate Pushdown : filtres au niveau Parquet
   ✓ Column Pruning : lecture sélective de colonnes
   ✓ Partition Pruning : lecture sélective de dossiers
   ✓ Catalyst Optimizer : réorganise le plan

📦 PARQUET
   ✓ Format colonnaire (parfait pour l'analytique)
   ✓ Compression efficace (gain 5-10x vs CSV)
   ✓ Métadonnées riches (schéma, stats)
   ✓ Partitionnem